# Homework 1 - 3008ICT

 ### No submission is required.


In [2]:
import torch
from jax import numpy as jnp
from flax import nnx
import time


Rngs = nnx.Rngs(0)

## 1. Speedtest for vectorization

Your goal is to measure the speed of linear algebra operations for different levels of vectorization. 
1. Construct two matrices $A$ and $B$ with Gaussian random entries of size $4096 \times 4096$. 
1. Compute $C = A B$ using matrix-matrix operations and report the time. 
1. Compute $C = A B$, treating $A$ as a matrix but computing the result for each column of $B$ one at a time. Report the time.
1. Compute $C = A B$, treating $A$ and $B$ as collections of vectors. Report the time.
1. Optional - what changes if you execute this on a GPU?

In [ ]:
## 1

size = 3

A = Rngs.normal((size, size))
B = Rngs.normal((size, size))

## 2

C = A @ B

## 3

C2 = jnp.zeros_like(A)

for i, col in enumerate(B.T):
    C2 = C2.at[:, i].set(A @ col)

assert jnp.absolute(C - C2).max() < 1e-4

## 4

C3 = jnp.zeros_like(A)

for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        C3 = C3.at[i,j].set(A[i,:] @ B[:,j])

assert jnp.absolute(C - C3).max() < 1e-4

## 5

# 2 will get faster.
# 3 and 4 will probably get slower

In [12]:
K = torch.arange(10)
K

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

## 2. NDArray and NumPy 

Your goal is to measure the speed penalty between Pytorch and Python when converting data between both. We are going to do this as follows:

1. Create two Gaussian random matrices $A, B$ of size $4096 \times 4096$ in PyTorch. 
1. Compute a vector $\mathbf{c} \in \mathbb{R}^{4096}$ where $c_i = \|A B_{i\cdot}\|^2$ where $\mathbf{c}$ is a **NumPy** vector.

To see the difference in speed due to Python perform the following two experiments and measure the time:

1. Compute $\|A B_{i\cdot}\|^2$ one at a time and assign its outcome to $\mathbf{c}_i$ directly.
1. Use an intermediate storage vector $\mathbf{d}$ in NDArray for assignments and copy to NumPy at the end.

## 3. Memory efficient computation

We want to compute $C \leftarrow A \cdot B + C$, where $A, B$ and $C$ are all matrices. Implement this in the most memory efficient manner. Pay attention to the following two things:

1. Do not allocate new memory for the new value of $C$.
1. Do not allocate new memory for intermediate results if possible.

## 4. Broadcast Operations

In order to perform polynomial fitting we want to compute a design matrix $A$ with 

$$A_{ij} = x_i^j$$

Our goal is to implement this **without a single for loop** entirely using vectorization and broadcast. 
    Here $1 \leq j \leq 20$ and $x = \{-10, -9.9, \ldots 10\}$. Implement code that generates such a matrix. 
    Note that $x_i^j$ is the result of $x_i$ to the power of $j$. 
